# Transcripción de Clases de Teología

Usa Whisper large-v3 con aceleración Metal (Apple Silicon) para transcribir audios de clase a texto.

In [1]:
# Instalar dependencias (solo la primera vez)
!pip install mlx-whisper mutagen

In [2]:
import mlx_whisper
from pathlib import Path
from mutagen import File as MutagenFile
import time

MODEL = "mlx-community/whisper-large-v3-turbo"

print(f"Modelo: {MODEL}")
print("Aceleración: Apple Metal (GPU)")

/Users/delpiwma/Documents/Projects/teologia/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modelo: mlx-community/whisper-large-v3-turbo
Aceleración: Apple Metal (GPU)


In [3]:
# Configuración
AUDIO_DIR = Path("./audios")
OUTPUT_DIR = Path("./transcripciones")
OUTPUT_DIR.mkdir(exist_ok=True)

EXTENSIONS = {".mp3", ".m4a", ".wav", ".ogg", ".flac", ".mp4", ".webm"}

audios = sorted([f for f in AUDIO_DIR.iterdir() if f.suffix.lower() in EXTENSIONS])

def get_duration(path):
    """Obtener duración del audio en segundos."""
    try:
        audio = MutagenFile(path)
        return audio.info.length if audio and audio.info else 0
    except:
        return 0

def fmt_duration(seconds):
    """Formatear segundos a HH:MM:SS."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h > 0:
        return f"{h}h {m:02d}m {s:02d}s"
    return f"{m}m {s:02d}s"

# Info de archivos
total_duration = 0
print(f"{'#':<4} {'Archivo':<40} {'Duración':<12} {'Estado'}")
print("─" * 70)
for idx, a in enumerate(audios, 1):
    dur = get_duration(a)
    total_duration += dur
    exists = "✅ ya transcrito" if (OUTPUT_DIR / f"{a.stem}.txt").exists() else "⏳ pendiente"
    print(f"{idx:<4} {a.name:<40} {fmt_duration(dur):<12} {exists}")

print("─" * 70)
print(f"Total: {len(audios)} archivos | {fmt_duration(total_duration)} de audio")

#    Archivo                                  Duración     Estado
──────────────────────────────────────────────────────────────────────
1    2026-03-06 18-44-39.m4a                  1h 35m 15s   ⏳ pendiente
2    2026-03-13 18-40-15.m4a                  1h 54m 48s   ⏳ pendiente
3    2026-03-20 18-37-08.m4a                  1m 30s       ⏳ pendiente
4    2026-03-20 18-38-41.m4a                  21m 53s      ⏳ pendiente
5    2026-03-20 19-00-36.m4a                  1h 38m 52s   ⏳ pendiente
6    2026-03-27 18-36-29.m4a                  1h 58m 27s   ⏳ pendiente
7    2026-04-10 18-36-32.m4a                  2h 00m 35s   ⏳ pendiente
8    2026-04-17 18-35-47.m4a                  2h 09m 36s   ⏳ pendiente
9    2026-04-24 18-36-39.m4a                  1h 58m 25s   ⏳ pendiente
10   2026-04-24 18-37-15.m4a                  1h 57m 39s   ✅ ya transcrito
11   2026-05-08 18-34-21.m4a                  1h 59m 31s   ✅ ya transcrito
12   2026-05-08 20-44-43.m4a                  2m 20s       ⏳ pendiente
───

In [4]:
# Transcribir con progreso
pending = [(a, get_duration(a)) for a in audios if not (OUTPUT_DIR / f"{a.stem}.txt").exists()]

if not pending:
    print("✅ Todos los audios ya están transcritos.")
else:
    total_pending_dur = sum(d for _, d in pending)
    elapsed_total = 0
    
    print(f"\n🎙️  Transcribiendo {len(pending)} archivos ({fmt_duration(total_pending_dur)})\n")
    
    for idx, (audio_path, duration) in enumerate(pending, 1):
        output_path = OUTPUT_DIR / f"{audio_path.stem}.txt"
        
        # Progreso
        pct = (idx - 1) / len(pending) * 100
        bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
        
        # ETA
        if elapsed_total > 0 and idx > 1:
            avg_ratio = elapsed_total / sum(d for _, d in pending[:idx-1])
            remaining_dur = sum(d for _, d in pending[idx-1:])
            eta = fmt_duration(remaining_dur * avg_ratio)
        else:
            eta = "calculando..."
        
        print(f"[{bar}] {pct:.0f}% | ETA: {eta}")
        print(f"  → {audio_path.name} ({fmt_duration(duration)})")
        
        start = time.time()
        result = mlx_whisper.transcribe(
            str(audio_path),
            path_or_hf_repo=MODEL,
            language="es",
        )
        took = time.time() - start
        elapsed_total += took
        
        output_path.write_text(result["text"], encoding="utf-8")
        speed = duration / took if took > 0 else 0
        print(f"  ✅ Listo en {fmt_duration(took)} ({speed:.1f}x realtime) | {len(result['text'])} chars\n")
    
    # Final
    bar = "█" * 20
    print(f"[{bar}] 100%")
    print(f"\n🏁 Completo. {len(pending)} archivos en {fmt_duration(elapsed_total)}")
    print(f"   Velocidad promedio: {total_pending_dur / elapsed_total:.1f}x realtime")


🎙️  Transcribiendo 10 archivos (13h 41m 46s)

[░░░░░░░░░░░░░░░░░░░░] 0% | ETA: calculando...
  → 2026-03-06 18-44-39.m4a (1h 35m 15s)


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 66841.50it/s]


  ✅ Listo en 2m 52s (33.2x realtime) | 77526 chars

[██░░░░░░░░░░░░░░░░░░] 10% | ETA: 21m 53s
  → 2026-03-13 18-40-15.m4a (1h 54m 48s)
  ✅ Listo en 3m 40s (31.2x realtime) | 97962 chars

[████░░░░░░░░░░░░░░░░] 20% | ETA: 19m 03s
  → 2026-03-20 18-37-08.m4a (1m 30s)
  ✅ Listo en 0m 03s (30.1x realtime) | 1374 chars

[██████░░░░░░░░░░░░░░] 30% | ETA: 19m 01s
  → 2026-03-20 18-38-41.m4a (21m 53s)
  ✅ Listo en 0m 41s (31.6x realtime) | 18974 chars

[████████░░░░░░░░░░░░] 40% | ETA: 18m 22s
  → 2026-03-20 19-00-36.m4a (1h 38m 52s)
  ✅ Listo en 3m 01s (32.7x realtime) | 74444 chars

[██████████░░░░░░░░░░] 50% | ETA: 15m 11s
  → 2026-03-27 18-36-29.m4a (1h 58m 27s)
  ✅ Listo en 3m 29s (34.0x realtime) | 95233 chars

[████████████░░░░░░░░] 60% | ETA: 11m 21s
  → 2026-04-10 18-36-32.m4a (2h 00m 35s)
  ✅ Listo en 3m 29s (34.5x realtime) | 96865 chars

[██████████████░░░░░░] 70% | ETA: 7m 34s
  → 2026-04-17 18-35-47.m4a (2h 09m 36s)
  ✅ Listo en 3m 56s (32.9x realtime) | 110568 chars

[██████████